# Build a RAG System with Sources

## Overview
- This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) system that includes source context
- It extends the contextual retrieval approach by providing both answers and the source documents used to generate them

## Key Functionalities

- **Document Loading & Processing**: Loads and chunks documents (PDFs, Wikipedia articles) for retrieval
- **Contextual Retrieval**: Uses similarity search to find relevant document chunks based on user queries
- **RAG Pipeline**: Combines retrieval with generation to produce contextual answers
- **Source Attribution**: Returns both the generated answer and the source documents used
- **Result Display**: Formats output to show query, response, and source context clearly

## Technical Components

- **Retrieval Chain**: `src_rag_response_chain` processes context and generates answers
- **Main Pipeline**: `rag_chain_w_sources` orchestrates retrieval and response generation
- **Document Formatting**: `format_docs()` converts document lists to searchable strings
- **Display Functions**: `display_results()` shows formatted results with source metadata

## Use Cases

- **Research & Analysis**: Get answers with traceable sources
- **Content Verification**: Check which documents support generated responses
- **Educational Applications**: Learn from both answers and source materials
- **Audit Trails**: Maintain transparency in AI-generated content

## Why We Need RAG with Sources

### **Transparency & Trust**
- **Verifiable Information**: Users can see exactly which documents were used to generate answers
- **Reduced Hallucinations**: Grounds responses in actual source material rather than model knowledge
- **Audit Trail**: Provides traceability for compliance and verification purposes

### **Quality Assurance**
- **Source Validation**: Users can independently verify the accuracy of generated responses
- **Context Understanding**: Shows the full context from which answers were derived
- **Confidence Assessment**: Helps users gauge reliability based on source quality

### **Educational & Research Value**
- **Deep Learning**: Users can explore source materials for comprehensive understanding
- **Citation Building**: Enables proper academic and professional referencing
- **Knowledge Discovery**: Reveals additional relevant information beyond the direct answer

### **Professional Applications**
- **Legal & Compliance**: Critical for regulatory requirements and legal documentation
- **Healthcare**: Essential for medical decision support with traceable evidence
- **Financial Services**: Required for audit trails and regulatory compliance
- **Academic Research**: Necessary for scholarly work and peer review

### **User Experience**
- **Informed Decisions**: Users can make better decisions with full context
- **Source Exploration**: Enables follow-up research and deeper investigation
- **Customization**: Users can choose which sources to trust or explore further

For this the section till `rag_prompt_template` would be same like previous `2. Build a Contextual Retrieval based RAG System` Notebook.

So we call that notebook and later customize the RAG_Pipeline to get the source context along with answer from RAG Pipeline

In [ ]:
%run '2. Build a Contextual Retrieval based RAG System.ipynb'

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda
from operator import itemgetter

chatgpt = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# src_rag_response_chain is a chain that:
# 1. Takes a dictionary with 'context' and 'question' keys.
# 2. Formats the 'context' (list of docs) into a single string using format_docs.
# 3. Passes the formatted context and question into the rag_prompt_template.
# 4. Sends the prompt to the chatgpt LLM.
# 5. Parses the LLM output into a string.
src_rag_response_chain = (
    {
        "context": (itemgetter('context') | RunnableLambda(format_docs)),
        "question": itemgetter("question")
    }
    | rag_prompt_template
    | chatgpt
    | StrOutputParser()
)

# rag_chain_w_sources is a higher-level chain that:
# 1. Retrieves relevant documents using similarity_retriever for the 'context'.
# 2. Passes the original question through unchanged.
# 3. Runs src_rag_response_chain to generate the answer, and assigns the result to a new 'response' key.
# This allows you to get both the answer and the source context used for the answer.
rag_chain_w_sources = (
    {
        "context": similarity_retriever,
        "question": RunnablePassthrough()
    }
    | RunnablePassthrough.assign(response=src_rag_response_chain)
)

In [ ]:
query = "What is machine learning?"
result = rag_chain_w_sources.invoke(query)
result

In [ ]:
from IPython.display import display, Markdown

def display_results(result_obj):
    print('Query:')
    display(Markdown(result_obj['question']))
    print()
    print('Response:')
    display(Markdown(result_obj['response']))
    print('='*50)
    print('Sources:')
    for source in result_obj['context']:
        print('Metadata:', source.metadata)
        print('Content Brief:')
        display(Markdown(source.page_content))
        print()


In [ ]:
query = "What is machine learning?"
result = rag_chain_w_sources.invoke(query)
display_results(result)

In [ ]:
query = "What is the difference between AI, ML and DL?"
result = rag_chain_w_sources.invoke(query)
display_results(result)

In [ ]:
query = "What is the difference between transformers and vision transformers?"
result = rag_chain_w_sources.invoke(query)
display_results(result)

In [ ]:
query = "What is an Agentic AI System?"
result = rag_chain_w_sources.invoke(query)
display_results(result)